# MPS topology and Dmax convergence

Compare routed CPU MPS with an exact 10-qubit long-range circuit and inspect automated convergence evidence.

## What you will learn

- How to express this workflow with Qiskit's reference simulator.
- How to change only the execution target to MettleQ.
- How correctness is checked before comparing timings.
- How to decide whether this workload is large enough to benefit from Apple-native execution.

## Performance model: what is actually being compared?

The reference simulator is **already running on this Mac's Apple CPU**. MettleQ is not comparing Apple Silicon with a machine that ignores it. Its opportunity is to reduce state-evolution cost through its MLX/Metal path, while paying extra planning, adapter, dispatch, synchronization, and result-conversion overhead.

Consequently, small circuits should often be faster on the SDK reference. MettleQ becomes useful only when the simulated state or repeated workload is large enough to amortize that overhead. The final result says which path won this particular measurement; it never assumes MettleQ won.

## Imports and measurement helpers

The SDK imports define the circuit and reference simulator. MettleQ's adapter supplies the alternate backend/device. The shared helpers make timing and numerical checks identical across the suite.

In [1]:
import numpy as np
from qiskit import QuantumCircuit, transpile
from qiskit.quantum_info import SparsePauliOp, Statevector
from qiskit.primitives import StatevectorEstimator, StatevectorSampler

from mettleq.integrations.qiskit import (
    MettleQBackend,
    MettleQEstimatorV2,
    MettleQSamplerV2,
)
from tutorials._support import (
    benchmark,
    emit_result,
    max_abs_error,
    phase_aligned_statevector_error,
    print_scaling_table,
    qiskit_selection,
    total_variation_distance,
)

## 1. Define the quantum problem

MPS cost depends on entanglement and routing-induced bond growth, not qubit count alone. Dmax caps the retained bond dimension.

In [2]:
rng = np.random.default_rng(41)
circuit = QuantumCircuit(10)
for layer in range(3):
    for wire in range(10):
        circuit.ry(float(rng.uniform(-1, 1)), wire)
    order = rng.permutation(10)
    for index in range(0, 10, 2):
        circuit.rzz(float(rng.uniform(-0.8, 0.8)), int(order[index]), int(order[index + 1]))

## 2. Run and time the SDK reference

This is the baseline a user would normally run. `benchmark` performs an unmeasured warm-up, synchronizes lazy results, and reports the median of repeated complete calls—not just a selected kernel.

In [3]:
reference, reference_ms, _ = benchmark(lambda: np.asarray(Statevector.from_instruction(circuit).data))

## 3. Run the same problem with MettleQ

Only the execution target changes. MettleQ records whether it selected exact statevector or MPS and whether that method ran on CPU or GPU. The candidate result is timed under the same warm-up and repeat policy.

In [4]:
backend = MettleQBackend(
    method="matrix_product_state",
    device="cpu",
    mps_max_bond_dimension=32,
    mps_truncation_threshold=1e-12,
    mps_routing_strategy="lookahead",
)
compiled = transpile(circuit, backend, optimization_level=1)

def run_mps():
    return np.asarray(backend.run(compiled, shots=1, return_statevector=True, execution_report=True).result().data(0)["statevector"])

candidate, mettleq_ms, _ = benchmark(run_mps)
error = phase_aligned_statevector_error(reference, candidate)
diagnostics = backend.last_mps_diagnostics[-1]
accuracy = backend.last_mps_accuracy_reports[-1]
convergence_estimator = MettleQEstimatorV2(
    method="matrix_product_state",
    device="cpu",
    mps_max_bond_dimension=32,
    mps_truncation_threshold=1e-12,
    mps_convergence_bond_dimensions=(8, 16, 32),
    mps_convergence_atol=5e-5,
)
convergence_result = convergence_estimator.run([(circuit, SparsePauliOp("IIIIIIIIIZ"))]).result()[0]
convergence = convergence_result.metadata["mettleq_mps_convergence"]
method, device = qiskit_selection(backend)

## 4. Check correctness before discussing speed

The MPS state is checked against an exact reference, local accuracy telemetry must pass, and successive Dmax values must converge.

In [5]:
tutorial_result = emit_result(
    notebook="qiskit/14_mps_topology_and_convergence.ipynb",
    framework="qiskit",
    reference_ms=reference_ms,
    mettleq_ms=mettleq_ms,
    check="phase-aligned MPS state atol=8e-5 and convergence report",
    passed=error <= 8e-5 and convergence["converged"] and accuracy["passed"],
    exact_match=bool(np.array_equal(reference, candidate)),
    selected_method=method,
    selected_device=device,
    metrics={"max_amplitude_error": error, "peak_bond": diagnostics["maximum_bond_dimension_reached"], "accuracy": accuracy, "convergence": convergence},
)


Comparison summary
------------------
Correctness contract: PASS — phase-aligned MPS state atol=8e-5 and convergence report
SDK reference median: 0.869 ms
MettleQ median:       12.102 ms
Timing interpretation: the SDK reference was 13.926x faster in this run.
MettleQ selected: matrix_product_state / cpu
Byte-for-byte result equality: no (see the declared tolerance/statistical check)

Machine-readable record (used by the suite runner):
TUTORIAL_RESULT::{"check": "phase-aligned MPS state atol=8e-5 and convergence report", "exact_match": false, "framework": "qiskit", "machine": "arm64", "metrics": {"accuracy": {"classification": "within_configured_local_thresholds", "observed": {"maximum_bond_dimension_reached": 32, "relative_discarded_weight_max": 1.2462929104416264e-24, "relative_discarded_weight_sum": 3.793931220328968e-24, "state_norm_error": 7.997455031549805e-09, "truncated": true}, "passed": true, "policy": "report", "schema_version": 1, "scope_warning": "Passing local telemetry t

## What should you conclude?

MPS is valuable for wide, weakly entangled circuits. It is not a general GPU speedup and approximation evidence must be inspected.

Read the output in this order:

1. **Correctness contract** must pass. A fast wrong result is not useful.
2. **Reference / MettleQ ratio** above `1.0×` means MettleQ was faster; below `1.0×` means the SDK reference was faster.
3. **Selected method/device** explains whether MettleQ used statevector or MPS and CPU or GPU.
4. Treat this notebook as a reproducible observation on this Mac, not a universal performance claim.